# OP-FL v2 — 01. Données V1 (eICU-CRD) — version 1.1

Ce notebook reconstruit **à zéro** le jeu de données de la contribution, avec un protocole sans fuite.
Chaque étape tient dans **une seule cellule** ; exécutez-les dans l'ordre.

| Étape | Contenu | Sorties principales |
|---|---|---|
| 0 | Configuration, chemins, outils communs | — |
| 1 | Cohorte (critères patients, un séjour éligible par patient) | `interim/cohort.parquet` |
| 2 | Laboratoire (16 variables) + créatinine pour l'AKI | `interim/lab_hourly.parquet`, `interim/creatinine.parquet` |
| 3 | Signes vitaux automatiques (`vitalPeriodic`) | `interim/vital_hourly.parquet` |
| 4 | Données infirmières : GCS (+ sédation) et température | `interim/gcs_hourly.parquet`, `interim/temp_hourly.parquet` |
| 5 | Diurèse (`intakeOutput`) | `interim/urine_hourly.parquet` |
| 6 | Qualité des données par hôpital (sans label) | `interim/cohort_final.parquet`, `reports/hospital_quality.csv` |
| 7 | Protocole figé : plis *leave-hospitals-out* + test verrouillé + splits patients | `splits/protocol.json`, `splits/patient_splits.parquet` |
| 8 | Labels : mortalité, AKI, durée de séjour, référence APACHE IVa | `interim/outcomes.parquet` |
| 9 | Tenseurs **bruts** par hôpital (X, M, C, Δ, S) | `clients/hospital_<id>.npz` |
| 10 | Normalisation par pli + disponibilité des variables par hôpital (sans label) | `normalizers/` |
| 11 | Module de chargement pour les notebooks suivants | `src/opfl_data.py` |
| 12 | Contrôles de cohérence | — |
| 13 | Couverture par hôpital + fiche de données | `reports/`, `DATA_CARD.md` |

**Changements de la version 1.1** (après la première exécution sur eICU)
* Cohorte : critères d'éligibilité **puis** un séjour par patient (premier séjour *éligible*), au lieu de l'inverse.
* Nouvelle étape 6 : exclusion des hôpitaux sans données de laboratoire ou de monitorage (absence structurelle), **avant** le tirage du protocole.
* Disponibilité de chaque variable par hôpital (`avail`), renvoyée par le chargeur.
* Température : suppression des doublons °C/°F d'une même saisie infirmière.
* Diurèse : exclusion des libellés qui ne sont pas des volumes (« Urine Count », « Occurrence », « Stool »).

**Principes**
1. Les données sont stockées **brutes** (non normalisées, sans forward-fill). La normalisation est appliquée au chargement, avec des statistiques calculées **uniquement sur le split train des hôpitaux d'entraînement du pli**.
2. Le protocole est tiré une seule fois puis **vérifié à chaque exécution** : il ne peut pas être re-tiré par accident.
3. Aucun critère de sélection et aucune statistique utilisée en entrée du modèle ne dépend des labels.

## Étape 0 — Configuration et outils communs

Adaptez uniquement `EICU_DIR` (dossier contenant les fichiers `.csv.gz` d'eICU-CRD v2.0).
Les bornes de plausibilité servent à retirer les erreurs de saisie (valeurs impossibles).

In [1]:
# ============================================================
# ÉTAPE 0 — CONFIGURATION, CHEMINS ET OUTILS COMMUNS
# ============================================================
import json, time, hashlib, sys, importlib
from collections import Counter
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)

# ---------- Chemins (à adapter) ----------
EICU_DIR    = Path("../../../dataset")   # dossier des fichiers eICU-CRD v2.0
PROJECT_DIR = Path("..")              # racine OP-FL-v2 (ce notebook est dans notebooks/)
DATA_DIR    = PROJECT_DIR / "data_v1"
SRC_DIR     = PROJECT_DIR / "src"

INTERIM, CLIENTS, SPLITS, NORMS, REPORTS = (DATA_DIR / d for d in
    ["interim", "clients", "splits", "normalizers", "reports"])
for d in [INTERIM, CLIENTS, SPLITS, NORMS, REPORTS, SRC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REQUIRED_TABLES = ["patient", "hospital", "lab", "vitalPeriodic",
                   "nurseCharting", "intakeOutput", "apachePatientResult"]

def table_path(name):
    for ext in (".csv.gz", ".csv"):
        p = EICU_DIR / f"{name}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"Table '{name}' introuvable dans {EICU_DIR.resolve()}")

missing = []
for t in REQUIRED_TABLES:
    try:
        table_path(t)
    except FileNotFoundError:
        missing.append(t)
if missing:
    raise FileNotFoundError(f"Tables manquantes dans {EICU_DIR.resolve()} : {missing}")

# ---------- Protocole ----------
SEED = 42
MIN_AGE = 18
MIN_ICU_MIN = 1440               # durée en unité >= 24 h
ONE_STAY_PER_PATIENT = True      # un seul séjour par patient : le premier séjour ÉLIGIBLE
MIN_STAYS_DEV = 500              # hôpitaux de développement (clients fédérés)
MIN_STAYS_LOCKED = 100           # hôpitaux [100, 500[ -> test final verrouillé
CORE_MIN_COVERAGE = 0.50         # hôpital retenu si >= 50 % de ses séjours ont un labo de routine ET une FC monitorée
AVAIL_MIN_COVERAGE = 0.05        # variable "disponible" dans un hôpital si >= 5 % des séjours (train) l'ont
N_FOLDS = 5                      # validation croisée leave-hospitals-out
SPLIT_RATIOS = (0.7, 0.1, 0.2)   # train / val / test au sein de chaque hôpital

# ---------- Fenêtre temporelle ----------
WINDOW_MIN = 1440                # 24 h d'observation
BIN_MIN = 60                     # pas horaire
T = WINDOW_MIN // BIN_MIN        # 24 pas de temps
AKI_HORIZON_MIN = 7 * 1440       # pic de créatinine cherché dans [24 h, 7 j]
CHUNK = 2_000_000                # lignes lues par bloc

# ---------- Variables : nom eICU -> (borne basse, borne haute) ----------
LAB_VARS = {
    "lactate": (0.1, 30), "creatinine": (0.1, 25), "BUN": (1, 300),
    "sodium": (100, 190), "potassium": (1.5, 10), "bicarbonate": (2, 60),
    "glucose": (10, 2000), "WBC x 1000": (0.05, 500), "Hgb": (2, 25),
    "Hct": (5, 75), "platelets x 1000": (1, 2000), "total bilirubin": (0.1, 70),
    "pH": (6.5, 8.0), "paO2": (10, 800), "paCO2": (5, 200), "FiO2": (21, 100),
}
BLOODGAS_LABS = ["lactate", "pH", "paO2", "paCO2", "FiO2"]
ROUTINE_LABS = [v for v in LAB_VARS if v not in BLOODGAS_LABS]
VITAL_VARS = {
    "heartrate": (20, 300), "sao2": (40, 100), "respiration": (2, 80),
    "systemicsystolic": (30, 300), "systemicdiastolic": (10, 200), "systemicmean": (20, 250),
}
GCS_VARS = {"GCS Total": (3, 15), "Eyes": (1, 4), "Motor": (1, 6), "Verbal": (1, 5)}
GCS_UNABLE_TEXT = "unable to score"
TEMP_RANGE = (30.0, 43.0)
TEMP_MIN_COVERAGE = 0.50         # température gardée si >= 50 % des séjours (train, dév.)
URINE_INCLUDE = r"urine|foley|void"
URINE_EXCLUDE = r"count|occurrence|stool|#"   # libellés qui ne sont pas des volumes
URINE_RANGE = (0.0, 2000.0)      # ml par saisie
HEIGHT_RANGE, WEIGHT_RANGE = (120, 230), (30, 300)
LOG_FEATURES = {"lactate", "creatinine", "BUN", "glucose", "WBC x 1000",
                "platelets x 1000", "total bilirubin", "urine_output"}

ALL_VAR_NAMES = (list(LAB_VARS) + list(VITAL_VARS) + list(GCS_VARS)
                 + ["gcs_unable", "temperature", "urine_output"])

# ---------- Outils communs ----------
def iter_chunks(name, usecols, stay_ids, chunksize=CHUNK):
    """Lit une table eICU par blocs et ne garde que les séjours de la cohorte."""
    t0 = time.time()
    for i, chunk in enumerate(pd.read_csv(table_path(name), usecols=usecols,
                                          chunksize=chunksize, low_memory=False)):
        print(f"  {name} : bloc {i + 1} ({time.time() - t0:.0f} s)", end="\r")
        yield chunk[chunk["patientunitstayid"].isin(stay_ids)]
    print(f"  {name} : lecture terminée en {time.time() - t0:.0f} s        ")

def to_hr(offset):
    return (offset // BIN_MIN).astype("int16")

def apply_range(df, ranges, qc):
    """Retire les valeurs hors bornes (les NaN, ex. GCS non évaluable, sont conservés)."""
    lo = df["var"].map({k: v[0] for k, v in ranges.items()}).astype(float)
    hi = df["var"].map({k: v[1] for k, v in ranges.items()}).astype(float)
    bad = df["value"].notna() & ((df["value"] < lo) | (df["value"] > hi))
    for v, n in df.loc[bad, "var"].astype(str).value_counts().items():
        qc[v] = qc.get(v, 0) + int(n)
    return df[~bad]

def partial_agg(df):
    """Agrégat partiel par (séjour, heure, variable) : somme, nb de valeurs, nb d'événements."""
    df = df.assign(var=pd.Categorical(df["var"], categories=ALL_VAR_NAMES))
    g = df.groupby(["patientunitstayid", "hr", "var"], sort=False, observed=True)["value"]
    out = pd.DataFrame({"vsum": g.sum(min_count=1), "n_val": g.count(), "n_obs": g.size()})
    return out.reset_index()

def finalize_agg(parts, how="mean"):
    """Combine les agrégats partiels (un séjour peut être coupé entre deux blocs)."""
    parts = [p for p in parts if len(p)]
    df = pd.concat(parts, ignore_index=True)
    df["var"] = pd.Categorical(df["var"].astype(str), categories=ALL_VAR_NAMES)
    df = (df.groupby(["patientunitstayid", "hr", "var"], sort=False, observed=True)
            [["vsum", "n_val", "n_obs"]].sum(min_count=1).reset_index())
    df["n_val"] = df["n_val"].fillna(0).astype("int32")
    df["n_obs"] = df["n_obs"].fillna(0).astype("int32")
    vsum = df["vsum"].to_numpy(dtype=float)
    if how == "mean":
        df["value"] = np.where(df["n_val"] > 0, vsum / df["n_val"].clip(lower=1), np.nan)
    else:
        df["value"] = np.where(df["n_val"] > 0, vsum, np.nan)
    return df[["patientunitstayid", "hr", "var", "value", "n_obs"]]

def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

print("Données eICU :", EICU_DIR.resolve())
print("Sorties     :", DATA_DIR.resolve())

Données eICU : C:\Users\Stryve\Desktop\StryveSolution\Imagerie-Phd\P.H.D\New Impl\dataset
Sorties     : C:\Users\Stryve\Desktop\StryveSolution\Imagerie-Phd\P.H.D\New Impl\FUALA-w_28-06-2026\OP-FL-V2\data_v1


## Étape 1 — Cohorte

Critères d'éligibilité, dans cet ordre :
1. première unité du séjour hospitalier (`unitvisitnumber == 1`) ;
2. âge ≥ 18 ans (`"> 89"` codé 90) ;
3. durée en unité ≥ 24 h ;
4. statut de sortie hospitalière connu ;
5. **un seul séjour par patient** (`uniquepid`) : le premier séjour *éligible* (tri par année de sortie puis identifiants) ;
6. hôpitaux ayant au moins 100 séjours : ≥ 500 → **développement**, 100 à 499 → **test final verrouillé**.

Les hôpitaux sans données de laboratoire ou de monitorage seront exclus à l'étape 6, une fois l'extraction faite.

In [2]:
# ============================================================
# ÉTAPE 1 — COHORTE
# ============================================================
patient = pd.read_csv(table_path("patient"), low_memory=False)

p = patient.copy()
p["age_num"] = pd.to_numeric(p["age"].astype(str).str.replace("> 89", "90", regex=False),
                             errors="coerce")
flow = []
def keep(df, label):
    flow.append((label, len(df), df["uniquepid"].nunique(), df["hospitalid"].nunique()))
    return df

p = keep(p, "Séjours en unité (eICU)")
p = keep(p[p["unitvisitnumber"] == 1], "1re unité du séjour hospitalier")
p = keep(p[p["age_num"] >= MIN_AGE], f"Âge >= {MIN_AGE} ans")
p = keep(p[p["unitdischargeoffset"] >= MIN_ICU_MIN], "Durée en unité >= 24 h")
p = keep(p[p["hospitaldischargestatus"].isin(["Alive", "Expired"])], "Statut de sortie connu")
p = p.sort_values(["uniquepid", "hospitaldischargeyear", "patienthealthsystemstayid",
                   "patientunitstayid"], na_position="last")
if ONE_STAY_PER_PATIENT:
    p = keep(p.drop_duplicates("uniquepid", keep="first"), "1er séjour éligible de chaque patient")
else:  # plusieurs séjours possibles, mais dans le premier hôpital du patient seulement
    first_h = p.groupby("uniquepid")["hospitalid"].first()
    p = keep(p[p["hospitalid"] == p["uniquepid"].map(first_h)], "Séjours éligibles dans le 1er hôpital du patient")

sizes = p.groupby("hospitalid").size()
role_of = pd.Series(np.select([sizes >= MIN_STAYS_DEV, sizes >= MIN_STAYS_LOCKED],
                              ["dev", "locked"], "excluded"), index=sizes.index)
p["role"] = p["hospitalid"].map(role_of)
p = keep(p[p["role"] != "excluded"], f"Hôpitaux >= {MIN_STAYS_LOCKED} séjours")

p["y_mortality"] = (p["hospitaldischargestatus"] == "Expired").astype("int8")
cols = ["patientunitstayid", "uniquepid", "patienthealthsystemstayid", "hospitalid", "role",
        "age_num", "gender", "ethnicity", "admissionheight", "admissionweight",
        "apacheadmissiondx", "unittype", "unitadmitsource", "unitdischargeoffset",
        "hospitaldischargeyear", "y_mortality"]
cohort = p[cols].sort_values(["hospitalid", "patientunitstayid"]).reset_index(drop=True)
cohort.to_parquet(INTERIM / "cohort.parquet", index=False)

flow_df = pd.DataFrame(flow, columns=["Étape", "Séjours", "Patients", "Hôpitaux"])
flow_df.to_csv(INTERIM / "flow_base.csv", index=False)
display(flow_df)
display(cohort.groupby("role").agg(hopitaux=("hospitalid", "nunique"), sejours=("patientunitstayid", "size")))

,Étape,Séjours,Patients,Hôpitaux
0,Séjours en unité (eICU),200859,139367,208
1,1re unité du séjour hospitalier,158442,133492,208
2,Âge >= 18 ans,157883,132994,208
3,Durée en unité >= 24 h,110257,95823,208
4,Statut de sortie connu,109218,95012,207
5,1er séjour éligible de chaque patient,95012,95012,207
6,Hôpitaux >= 100 séjours,93100,93100,158


,hopitaux,sejours
role,,
dev,61,68304
locked,97,24796


## Étape 2 — Laboratoire

Un seul passage sur `lab.csv.gz` :
* les 16 variables sur la fenêtre [0, 24 h[ → valeur moyenne et **nombre de mesures** par heure ;
* la créatinine jusqu'à 7 jours → baseline (minimum sur 24 h) et pic (maximum sur [24 h, 7 j]) pour l'AKI.

FiO2 saisie en fraction (0,21–1) convertie en % ; valeurs impossibles retirées et comptées.

In [3]:
# ============================================================
# ÉTAPE 2 — LABORATOIRE (16 VARIABLES + CRÉATININE POUR L'AKI)
# ============================================================
cohort = pd.read_parquet(INTERIM / "cohort.parquet")
STAY_IDS = cohort["patientunitstayid"].to_numpy()

parts, cr_rows, qc_drop = [], [], {}
qc = {"fio2_fraction_converted": 0}
for chunk in iter_chunks("lab", ["patientunitstayid", "labresultoffset", "labname", "labresult"], STAY_IDS):
    c = chunk[chunk["labname"].isin(list(LAB_VARS)) & chunk["labresult"].notna()
              & (chunk["labresultoffset"] >= 0) & (chunk["labresultoffset"] <= AKI_HORIZON_MIN)]
    if c.empty:
        continue
    c = c.rename(columns={"labname": "var", "labresult": "value"})
    frac = (c["var"] == "FiO2") & (c["value"] <= 1.0)
    qc["fio2_fraction_converted"] += int(frac.sum())
    c.loc[frac, "value"] = c.loc[frac, "value"] * 100
    c = apply_range(c, LAB_VARS, qc_drop)
    cr_rows.append(c.loc[c["var"] == "creatinine", ["patientunitstayid", "labresultoffset", "value"]])
    w = c[c["labresultoffset"] < WINDOW_MIN]
    w = w.assign(hr=to_hr(w["labresultoffset"]))[["patientunitstayid", "hr", "var", "value"]]
    parts.append(partial_agg(w))

lab_hourly = finalize_agg(parts, how="mean")
lab_hourly.to_parquet(INTERIM / "lab_hourly.parquet", index=False)

cr = pd.concat(cr_rows, ignore_index=True)
base = cr[cr["labresultoffset"] < WINDOW_MIN].groupby("patientunitstayid")["value"].min().rename("cr_base")
peak = cr[cr["labresultoffset"] >= WINDOW_MIN].groupby("patientunitstayid")["value"].max().rename("cr_peak")
creat = pd.concat([base, peak], axis=1).rename_axis("patientunitstayid").reset_index()
creat.to_parquet(INTERIM / "creatinine.parquet", index=False)

qc["dropped_out_of_range"] = qc_drop
save_json(qc, INTERIM / "qc_lab.json")

pres = (lab_hourly.groupby("var", observed=True)["patientunitstayid"].nunique() / len(STAY_IDS))
print("Lignes horaires :", len(lab_hourly))
print("FiO2 converties de fraction en % :", qc["fio2_fraction_converted"])
display(pd.DataFrame({"presence_24h": pres.round(3),
                      "valeurs_retirees": pd.Series(qc_drop)}).fillna(0).sort_values("presence_24h"))

  lab : lecture terminée en 63 s        
Lignes horaires : 1875814
FiO2 converties de fraction en % : 10644


,presence_24h,valeurs_retirees
lactate,0.274,13
pH,0.375,10
paCO2,0.378,12
paO2,0.382,10
FiO2,0.400,3075
total bilirubin,0.422,5
bicarbonate,0.866,3
platelets x 1000,0.883,6
WBC x 1000,0.890,63
Hgb,0.900,11


## Étape 3 — Signes vitaux automatiques (`vitalPeriodic`)

Agrégation horaire au format large. La PA « systemic » est la **PA invasive** : sa présence indique un cathéter artériel.
La température de cette table est mise de côté et fusionnée à l'étape 4 avec celle de `nurseCharting`.
Les fréquences respiratoires à 0 ou 1 (artefacts de monitorage) sont retirées par la borne basse.

In [4]:
# ============================================================
# ÉTAPE 3 — SIGNES VITAUX AUTOMATIQUES (vitalPeriodic)
# ============================================================
cohort = pd.read_parquet(INTERIM / "cohort.parquet")
STAY_IDS = cohort["patientunitstayid"].to_numpy()

VCOLS = list(VITAL_VARS) + ["temperature"]
RANGES = {**VITAL_VARS, "temperature": TEMP_RANGE}
parts, temp_parts, qc_drop = [], [], {}
for chunk in iter_chunks("vitalPeriodic", ["patientunitstayid", "observationoffset"] + VCOLS, STAY_IDS):
    c = chunk[(chunk["observationoffset"] >= 0) & (chunk["observationoffset"] < WINDOW_MIN)].copy()
    if c.empty:
        continue
    f = c["temperature"] > 50                      # saisie en °F
    c.loc[f, "temperature"] = (c.loc[f, "temperature"] - 32) * 5 / 9
    for v, (lo, hi) in RANGES.items():
        bad = c[v].notna() & ((c[v] < lo) | (c[v] > hi))
        qc_drop[v] = qc_drop.get(v, 0) + int(bad.sum())
        c.loc[bad, v] = np.nan
    c["hr"] = to_hr(c["observationoffset"])
    g = c.groupby(["patientunitstayid", "hr"])[VCOLS]
    s = g.sum(min_count=1).reset_index().melt(["patientunitstayid", "hr"], var_name="var", value_name="vsum")
    n = g.count().reset_index().melt(["patientunitstayid", "hr"], var_name="var", value_name="n_val")
    s["n_val"] = n["n_val"].to_numpy()
    s = s[s["n_val"] > 0].assign(n_obs=lambda d: d["n_val"])
    parts.append(s[s["var"] != "temperature"])
    temp_parts.append(s[s["var"] == "temperature"])

vital_hourly = finalize_agg(parts, how="mean")
vital_hourly.to_parquet(INTERIM / "vital_hourly.parquet", index=False)
temp_vital = pd.concat(temp_parts, ignore_index=True)
temp_vital["var"] = temp_vital["var"].astype(str)
temp_vital.to_parquet(INTERIM / "temp_partials_vital.parquet", index=False)
save_json({"dropped_out_of_range": qc_drop}, INTERIM / "qc_vital.json")

pres = vital_hourly.groupby("var", observed=True)["patientunitstayid"].nunique() / len(STAY_IDS)
pres["temperature (vitalPeriodic seule)"] = temp_vital["patientunitstayid"].nunique() / len(STAY_IDS)
qc_show = {("temperature (vitalPeriodic seule)" if k == "temperature" else k): v for k, v in qc_drop.items()}
print("Lignes horaires :", len(vital_hourly))
display(pd.DataFrame({"presence_24h": pres.round(3), "valeurs_retirees": pd.Series(qc_show)}).fillna(0))

  vitalPeriodic : lecture terminée en 245 s        
Lignes horaires : 7455881


,presence_24h,valeurs_retirees
heartrate,0.974,2610
sao2,0.967,3206
respiration,0.902,176784
systemicsystolic,0.268,5802
systemicdiastolic,0.268,9106
systemicmean,0.268,27306
temperature (vitalPeriodic seule),0.115,4970


## Étape 4 — Données infirmières : GCS, sédation et température

* **Sédation** : une saisie « Unable to score due to medication » compte comme un **événement d'observation** (M = 1, valeur manquante) et alimente la variable `gcs_unable` (1 = non évaluable, 0 = GCS total chiffré).
* **Température** : `vitalPeriodic` + `nurseCharting`. Dans eICU, une même saisie infirmière apparaît souvent deux fois, en °C et en °F : on ne garde qu'une ligne par (séjour, instant), de préférence celle en °C. L'inclusion de la température est décidée à l'étape 9 (couverture ≥ 50 % sur le train des hôpitaux de développement).

In [5]:
# ============================================================
# ÉTAPE 4 — DONNÉES INFIRMIÈRES : GCS (+ SÉDATION) ET TEMPÉRATURE
# ============================================================
cohort = pd.read_parquet(INTERIM / "cohort.parquet")
STAY_IDS = cohort["patientunitstayid"].to_numpy()

cols = ["patientunitstayid", "nursingchartoffset", "nursingchartcelltypevallabel",
        "nursingchartcelltypevalname", "nursingchartvalue"]
gcs_parts, temp_parts, qc_drop = [], [], {}
qc = Counter()
temp_names = Counter()
for chunk in iter_chunks("nurseCharting", cols, STAY_IDS):
    c = chunk[(chunk["nursingchartoffset"] >= 0) & (chunk["nursingchartoffset"] < WINDOW_MIN)]
    label = c["nursingchartcelltypevallabel"].astype(str)

    # ---- GCS ----
    g = c[(label == "Glasgow coma score") & c["nursingchartcelltypevalname"].isin(list(GCS_VARS))]
    qc["gcs_empty_dropped"] += int(g["nursingchartvalue"].isna().sum())
    g = g[g["nursingchartvalue"].notna()]
    txt = g["nursingchartvalue"].astype(str).str.strip()
    unable = txt.str.lower().str.contains(GCS_UNABLE_TEXT, regex=False).to_numpy()
    num = pd.to_numeric(txt, errors="coerce").to_numpy()
    other = ~unable & np.isnan(num)
    qc["gcs_unable"] += int(unable.sum()); qc["gcs_numeric"] += int((~np.isnan(num)).sum())
    qc["gcs_other_text_dropped"] += int(other.sum())
    ok = ~other
    comp = pd.DataFrame({"patientunitstayid": g["patientunitstayid"].to_numpy()[ok],
                         "hr": to_hr(g["nursingchartoffset"]).to_numpy()[ok],
                         "var": g["nursingchartcelltypevalname"].to_numpy()[ok],
                         "value": np.where(unable, np.nan, num)[ok]})
    comp = apply_range(comp, GCS_VARS, qc_drop)
    is_total_num = (g["nursingchartcelltypevalname"].to_numpy() == "GCS Total") & ~np.isnan(num)
    fl = unable | is_total_num
    flag = pd.DataFrame({"patientunitstayid": g["patientunitstayid"].to_numpy()[fl],
                         "hr": to_hr(g["nursingchartoffset"]).to_numpy()[fl],
                         "var": "gcs_unable", "value": unable[fl].astype(float)})
    gcs_parts += [partial_agg(comp), partial_agg(flag)]

    # ---- Température (une ligne par séjour et par instant, °C de préférence) ----
    t = c[label.str.contains("temperature", case=False, na=False)]
    name = t["nursingchartcelltypevalname"].astype(str)
    temp_names.update(name.value_counts().to_dict())
    is_c = name.str.contains("(C)", regex=False); is_f = name.str.contains("(F)", regex=False)
    val = pd.to_numeric(t["nursingchartvalue"], errors="coerce")
    val = val.where(~(is_f | (is_c & (val > 50))), (val - 32) * 5 / 9)
    sel = (is_c | is_f) & val.notna()
    tt = pd.DataFrame({"patientunitstayid": t["patientunitstayid"][sel], "offset": t["nursingchartoffset"][sel],
                       "pref": np.where(is_c[sel], 0, 1), "value": val[sel]})
    n_before = len(tt)
    tt = tt.sort_values("pref", kind="stable").drop_duplicates(["patientunitstayid", "offset"])
    qc["temperature_C_F_duplicates_removed"] += n_before - len(tt)
    tt = pd.DataFrame({"patientunitstayid": tt["patientunitstayid"], "hr": to_hr(tt["offset"]),
                       "var": "temperature", "value": tt["value"]})
    tt = apply_range(tt, {"temperature": TEMP_RANGE}, qc_drop)
    temp_parts.append(partial_agg(tt))

gcs_hourly = finalize_agg(gcs_parts, how="mean")
gcs_hourly.to_parquet(INTERIM / "gcs_hourly.parquet", index=False)

temp_vital = pd.read_parquet(INTERIM / "temp_partials_vital.parquet")
temp_hourly = finalize_agg(temp_parts + [temp_vital], how="mean")
temp_hourly.to_parquet(INTERIM / "temp_hourly.parquet", index=False)
save_json({**dict(qc), "dropped_out_of_range": qc_drop, "temperature_names": dict(temp_names)}, INTERIM / "qc_nursing.json")

print("GCS : saisies chiffrées", qc["gcs_numeric"], "| non évaluables (sédation)", qc["gcs_unable"],
      "| vides retirées", qc["gcs_empty_dropped"], "| autre texte retiré", qc["gcs_other_text_dropped"])
print("Libellés de température rencontrés :", dict(temp_names.most_common(8)))
print("Doublons °C/°F de température retirés :", qc["temperature_C_F_duplicates_removed"])
print(f"Séjours avec une température en 24 h : {temp_hourly['patientunitstayid'].nunique() / len(STAY_IDS):.1%}")

  nurseCharting : lecture terminée en 288 s        
GCS : saisies chiffrées 1750117 | non évaluables (sédation) 13965 | vides retirées 37863 | autre texte retiré 0
Libellés de température rencontrés : {'Temperature (C)': 826842, 'Temperature (F)': 826738, 'Temperature Location': 474655}
Doublons °C/°F de température retirés : 826663
Séjours avec une température en 24 h : 94.1%


## Étape 5 — Diurèse (`intakeOutput`)

Chemin « output » et libellé contenant urine / foley / void, **sauf** les libellés qui ne sont pas des volumes (« Urine Count », « Urine Occurrence », « Mixed Urine/Stool… »).
Valeurs négatives ou > 2 000 ml retirées. Valeur horaire = **somme** des volumes ; un volume de 0 ml est une vraie mesure.

In [6]:
# ============================================================
# ÉTAPE 5 — DIURÈSE (intakeOutput)
# ============================================================
cohort = pd.read_parquet(INTERIM / "cohort.parquet")
STAY_IDS = cohort["patientunitstayid"].to_numpy()

cols = ["patientunitstayid", "intakeoutputoffset", "cellpath", "celllabel", "cellvaluenumeric"]
parts, qc_drop, labels, excluded = [], {}, Counter(), Counter()
for chunk in iter_chunks("intakeOutput", cols, STAY_IDS):
    c = chunk[(chunk["intakeoutputoffset"] >= 0) & (chunk["intakeoutputoffset"] < WINDOW_MIN)
              & chunk["cellvaluenumeric"].notna()]
    is_out = c["cellpath"].str.contains("output", case=False, na=False)
    is_urine = c["celllabel"].str.contains(URINE_INCLUDE, case=False, na=False, regex=True)
    not_volume = c["celllabel"].str.contains(URINE_EXCLUDE, case=False, na=False, regex=True)
    excluded.update(c.loc[is_out & is_urine & not_volume, "celllabel"].value_counts().to_dict())
    c = c[is_out & is_urine & ~not_volume]
    labels.update(c["celllabel"].value_counts().to_dict())
    u = pd.DataFrame({"patientunitstayid": c["patientunitstayid"], "hr": to_hr(c["intakeoutputoffset"]),
                      "var": "urine_output", "value": c["cellvaluenumeric"].astype(float)})
    u = apply_range(u, {"urine_output": URINE_RANGE}, qc_drop)
    parts.append(partial_agg(u))

urine_hourly = finalize_agg(parts, how="sum")
urine_hourly.to_parquet(INTERIM / "urine_hourly.parquet", index=False)
save_json({"dropped_out_of_range": qc_drop, "top_labels": dict(labels.most_common(40)),
           "excluded_labels": dict(excluded.most_common(40))}, INTERIM / "qc_urine.json")

n_with = urine_hourly["patientunitstayid"].nunique()
print(f"Séjours avec au moins une diurèse en 24 h : {n_with} / {len(STAY_IDS)} ({n_with / len(STAY_IDS):.1%})")
print("Valeurs retirées :", qc_drop)
print("Libellés exclus (pas des volumes) :", dict(excluded.most_common(10)))
print("Libellés retenus les plus fréquents :")
display(pd.Series(dict(labels.most_common(15)), name="lignes"))

  intakeOutput : lecture terminée en 64 s        
Séjours avec au moins une diurèse en 24 h : 63295 / 93100 (68.0%)
Valeurs retirées : {'urine_output': 3397}
Libellés exclus (pas des volumes) : {'Urine Count': 1708, 'Urine Occurrence': 277, 'Mixed Urine/Stool Volume': 220, 'urine count': 19}
Libellés retenus les plus fréquents :


Urine                                    454145
URINE CATHETER                            43010
Foley cath                                 2148
Voided Amount                              1799
Foley                                       637
SN Urine Output(ml)                         372
foley                                       251
Urine Output-Foley                          225
Urine, void:                                178
Urine Output (mL)-Urethral Catheter         142
Urine Output (mL)-Urethral Catheter         129
Urine Output (mL)-Urethral Catheter          84
Urine Output-foley                           72
FOLEY                                        58
Suprapubic Urine Output                      45
Name: lignes, dtype: int64

## Étape 6 — Qualité des données par hôpital (sans label)

Dans eICU, certains hôpitaux n'ont pas interfacé un module entier (aucun laboratoire, ou presque pas de `vitalPeriodic`).
Dans ces hôpitaux, M = 0 ne reflète pas une décision clinique mais une **absence structurelle** : c'est un pur artefact de site.

Critère d'inclusion défini *a priori* et **sans label** : un hôpital est retenu si au moins 50 % de ses séjours ont (i) au moins un laboratoire de routine et (ii) une fréquence cardiaque monitorée dans les 24 h.
Il s'applique à tous les hôpitaux (développement et test verrouillé) **avant** le tirage du protocole.

Les modules partiellement disponibles (gaz du sang, PA invasive, GCS…) sont conservés ; leur disponibilité par hôpital est calculée à l'étape 10.

In [7]:
# ============================================================
# ÉTAPE 6 — QUALITÉ DES DONNÉES PAR HÔPITAL (SANS LABEL)
# ============================================================
cohort = pd.read_parquet(INTERIM / "cohort.parquet")
hospital = pd.read_csv(table_path("hospital"))
stay_h = cohort.set_index("patientunitstayid")["hospitalid"]
n_h = cohort.groupby("hospitalid").size()

pres = []
for n in ["lab_hourly", "vital_hourly", "gcs_hourly", "temp_hourly", "urine_hourly"]:
    d = pd.read_parquet(INTERIM / f"{n}.parquet", columns=["patientunitstayid", "var"])
    d["var"] = d["var"].astype(str)
    pres.append(d.drop_duplicates())
pres = pd.concat(pres, ignore_index=True)
pres = pres[pres["patientunitstayid"].isin(stay_h.index)]
pres["hospitalid"] = pres["patientunitstayid"].map(stay_h)

cov = (pres.groupby(["hospitalid", "var"])["patientunitstayid"].nunique().unstack(fill_value=0)
       .reindex(index=n_h.index, columns=ALL_VAR_NAMES, fill_value=0).div(n_h, axis=0))
lab_any = (pres[pres["var"].isin(ROUTINE_LABS)].groupby("hospitalid")["patientunitstayid"].nunique()
           .reindex(n_h.index, fill_value=0) / n_h)

quality = cov.add_prefix("cov_")
quality.insert(0, "cov_routine_lab_any", lab_any)
quality.insert(0, "role", cohort.groupby("hospitalid")["role"].first())
quality.insert(0, "n_stays", n_h)
quality["included"] = ((quality["cov_routine_lab_any"] >= CORE_MIN_COVERAGE)
                       & (quality["cov_heartrate"] >= CORE_MIN_COVERAGE))
quality = quality.reset_index().merge(hospital, on="hospitalid", how="left")
quality.to_csv(REPORTS / "hospital_quality.csv", index=False)

keep_h = quality.loc[quality["included"], "hospitalid"]
cohort_final = cohort[cohort["hospitalid"].isin(keep_h)].reset_index(drop=True)
cohort_final.to_parquet(INTERIM / "cohort_final.parquet", index=False)

excl = quality[~quality["included"]]
flow_df = pd.read_csv(INTERIM / "flow_base.csv")
flow_df.loc[len(flow_df)] = [f"Labo de routine et monitorage disponibles ({len(excl)} hôpitaux exclus)",
                             len(cohort_final), cohort_final["uniquepid"].nunique(), cohort_final["hospitalid"].nunique()]
flow_df.to_csv(REPORTS / "flowchart.csv", index=False)
display(flow_df)

print(f"Hôpitaux exclus (absence structurelle : < {CORE_MIN_COVERAGE:.0%} des séjours avec labo ou FC) :")
display(excl[["hospitalid", "role", "n_stays", "cov_routine_lab_any", "cov_heartrate"]].round(3))
display(cohort_final.groupby("role").agg(hopitaux=("hospitalid", "nunique"), sejours=("patientunitstayid", "size"),
                                         taille_min=("hospitalid", lambda s: s.value_counts().min()),
                                         taille_max=("hospitalid", lambda s: s.value_counts().max())))
print("Distribution des couvertures (hôpitaux retenus) :")
display(quality.loc[quality["included"], ["cov_routine_lab_any", "cov_heartrate", "cov_lactate",
                                          "cov_systemicmean", "cov_GCS Total", "cov_urine_output"]].describe().round(3))

,Étape,Séjours,Patients,Hôpitaux
0,Séjours en unité (eICU),200859,139367,208
1,1re unité du séjour hospitalier,158442,133492,208
2,Âge >= 18 ans,157883,132994,208
3,Durée en unité >= 24 h,110257,95823,208
4,Statut de sortie connu,109218,95012,207
5,1er séjour éligible de chaque patient,95012,95012,207
6,Hôpitaux >= 100 séjours,93100,93100,158
7,Labo de routine et monitorage disponibles (2 h...,90734,90734,156


Hôpitaux exclus (absence structurelle : < 50% des séjours avec labo ou FC) :


,hospitalid,role,n_stays,cov_routine_lab_any,cov_heartrate
64,246,locked,156,0.891,0.083
151,443,dev,2210,0.986,0.300


,hopitaux,sejours,taille_min,taille_max
role,,,,
dev,60,66094,521,3539
locked,96,24640,101,499


Distribution des couvertures (hôpitaux retenus) :


,cov_routine_lab_any,cov_heartrate,cov_lactate,cov_systemicmean,cov_GCS Total,cov_urine_output
count,156.000,156.000,156.000,156.000,156.000,156.000
mean,0.947,0.988,0.259,0.200,0.626,0.660
std,0.047,0.037,0.145,0.168,0.366,0.287
min,0.706,0.671,0.000,0.000,0.000,0.000
25%,0.931,0.990,0.153,0.062,0.344,0.467
50%,0.958,0.997,0.243,0.177,0.757,0.669
75%,0.978,1.000,0.348,0.290,0.973,0.951
max,1.000,1.000,0.736,0.824,1.000,1.000


## Étape 7 — Protocole figé

* **Développement** : les hôpitaux ≥ 500 séjours sont répartis en 5 plis *leave-hospitals-out*, équilibrés en taille.
* **Test final verrouillé** : les hôpitaux de 100 à 499 séjours. On ne les évalue **qu'une seule fois**, pour le tableau final de l'article.
* **Au sein de chaque hôpital** : split par patient 70 / 10 / 20, stratifié sur la mortalité.

| Rôle de l'hôpital dans un pli | `train` | `val` | `test` |
|---|---|---|---|
| Hôpital d'entraînement | apprentissage | sélection de modèle | évaluation intra-fédération |
| Hôpital tenu à l'écart (pli ou test verrouillé) | adaptation **sans label** (normalisation locale, propension) | évaluation | évaluation |

Si `protocol.json` existe déjà, la cellule vérifie que le tirage est identique au lieu de l'écraser.

In [8]:
# ============================================================
# ÉTAPE 7 — PROTOCOLE : PLIS LEAVE-HOSPITALS-OUT ET SPLITS PATIENTS (FIGÉS)
# ============================================================
cohort = pd.read_parquet(INTERIM / "cohort_final.parquet")
rng = np.random.default_rng(SEED)

hs = cohort.groupby(["hospitalid", "role"]).size().rename("n").reset_index()
dev_ids = hs[hs["role"] == "dev"].sort_values(["n", "hospitalid"], ascending=[False, True])["hospitalid"].to_numpy()
locked_ids = np.sort(hs[hs["role"] == "locked"]["hospitalid"].to_numpy())

# 1) plis équilibrés en taille : blocs de N_FOLDS hôpitaux consécutifs (triés par taille), permutés
fold_of = {}
for start in range(0, len(dev_ids), N_FOLDS):
    block = dev_ids[start:start + N_FOLDS]
    for h, f in zip(block, rng.permutation(N_FOLDS)[:len(block)]):
        fold_of[int(h)] = int(f)

# 2) splits par patient, stratifiés sur la mortalité, au sein de chaque hôpital
pat = (cohort.groupby(["hospitalid", "uniquepid"])["y_mortality"].max()
       .reset_index().sort_values(["hospitalid", "uniquepid"]))
pat["split"] = ""
for h in np.sort(pat["hospitalid"].unique()):
    rows = pat.index[pat["hospitalid"] == h]
    y = pat.loc[rows, "y_mortality"].to_numpy()
    lab = np.empty(len(rows), dtype=object)
    for cls in (0, 1):
        idx = rng.permutation(np.flatnonzero(y == cls))
        n_tr = int(round(SPLIT_RATIOS[0] * len(idx)))
        n_va = int(round(SPLIT_RATIOS[1] * len(idx)))
        lab[idx[:n_tr]], lab[idx[n_tr:n_tr + n_va]], lab[idx[n_tr + n_va:]] = "train", "val", "test"
    pat.loc[rows, "split"] = lab

splits = cohort[["patientunitstayid", "uniquepid", "hospitalid", "role", "y_mortality"]].merge(
    pat[["hospitalid", "uniquepid", "split"]], on=["hospitalid", "uniquepid"], how="left")
splits["fold"] = splits["hospitalid"].map(fold_of).fillna(-1).astype(int)

key = splits.sort_values("patientunitstayid")[["patientunitstayid", "split", "fold"]].astype(str).agg("|".join, axis=1)
digest = hashlib.sha256("\n".join(key).encode()).hexdigest()[:16]

protocol = {
    "hash": digest,
    "version": "1.1",
    "seed": SEED,
    "split_ratios": dict(zip(["train", "val", "test"], SPLIT_RATIOS)),
    "n_folds": N_FOLDS,
    "folds": {str(k): sorted(int(h) for h, f in fold_of.items() if f == k) for k in range(N_FOLDS)},
    "locked_test_hospitals": [int(h) for h in locked_ids],
    "criteria": {"min_age": MIN_AGE, "min_icu_minutes": MIN_ICU_MIN,
                 "one_stay_per_patient": "first eligible stay" if ONE_STAY_PER_PATIENT else False,
                 "min_stays_dev": MIN_STAYS_DEV, "min_stays_locked": MIN_STAYS_LOCKED,
                 "core_min_coverage": CORE_MIN_COVERAGE},
    "usage": {
        "training_hospital": {"train": "apprentissage", "val": "sélection de modèle",
                              "test": "évaluation intra-fédération"},
        "heldout_hospital": {"train": "adaptation SANS label (normalisation locale, propension)",
                             "val+test": "évaluation"},
        "locked_test": "évalué une seule fois, à la fin, avec le modèle entraîné sur tous les hôpitaux de dév.",
    },
    "created_at": datetime.now().isoformat(timespec="seconds"),
}

proto_path = SPLITS / "protocol.json"
if proto_path.exists():
    old = load_json(proto_path)
    if old["hash"] != digest:
        raise RuntimeError(
            f"Le protocole existant (hash {old['hash']}) diffère du tirage actuel ({digest}). "
            "La cohorte ou un paramètre a changé. Ne supprimez splits/ que si vous voulez "
            "volontairement re-tirer le protocole.")
    print(f"Protocole existant retrouvé à l'identique (hash {digest}) : rien n'est écrasé.")
    if not (SPLITS / "patient_splits.parquet").exists():
        splits.drop(columns="y_mortality").to_parquet(SPLITS / "patient_splits.parquet", index=False)
else:
    save_json(protocol, proto_path)
    splits.drop(columns="y_mortality").to_parquet(SPLITS / "patient_splits.parquet", index=False)
    print(f"Protocole tiré et figé (hash {digest}).")

summary = (splits.assign(group=np.where(splits["fold"] >= 0, "pli " + splits["fold"].astype(str), "test verrouillé"))
           .groupby("group").agg(hopitaux=("hospitalid", "nunique"), sejours=("patientunitstayid", "size"),
                                 deces=("y_mortality", "sum"), mortalite=("y_mortality", "mean")))
display(summary)
display(pd.crosstab(splits["role"], splits["split"], margins=True))

Protocole tiré et figé (hash d92099ea7c11d517).


,hopitaux,sejours,deces,mortalite
group,,,,
pli 0,12,12709,1156,0.090959
pli 1,12,12689,1254,0.098826
pli 2,12,13949,1343,0.096279
pli 3,12,12888,1267,0.098309
pli 4,12,13859,1172,0.084566
test verrouillé,96,24640,2029,0.082346


split,test,train,val,All
role,,,,
dev,13218,46268,6608,66094
locked,4930,17244,2466,24640
All,18148,63512,9074,90734


## Étape 8 — Labels

| Label | Définition |
|---|---|
| `y_mortality` | décès hospitalier |
| `y_aki` | KDIGO stade ≥ 2 simplifié : pic de créatinine [24 h, 7 j] / minimum sur 24 h ≥ 2 ; défini seulement si la créatinine est dosée aux deux périodes (`aki_ascertained`) |
| `y_los_gt7` | durée en unité > 7 jours |
| `apache_pred_mortality` | prédiction APACHE IVa : **référence clinique uniquement, jamais une entrée du modèle** |

In [9]:
# ============================================================
# ÉTAPE 8 — LABELS : MORTALITÉ, AKI, DURÉE DE SÉJOUR, RÉFÉRENCE APACHE IVa
# ============================================================
cohort = pd.read_parquet(INTERIM / "cohort_final.parquet")
creat = pd.read_parquet(INTERIM / "creatinine.parquet")

out = cohort[["patientunitstayid", "hospitalid", "role", "y_mortality", "unitdischargeoffset"]].merge(
    creat, on="patientunitstayid", how="left")
out["aki_ascertained"] = out["cr_base"].notna() & out["cr_peak"].notna()
out["y_aki"] = np.where(out["aki_ascertained"], (out["cr_peak"] / out["cr_base"] >= 2.0).astype(float), np.nan)
out["los_icu_days"] = out["unitdischargeoffset"] / 1440.0
out["y_los_gt7"] = (out["los_icu_days"] > 7).astype("int8")

apache = pd.read_csv(table_path("apachePatientResult"),
                     usecols=["patientunitstayid", "apacheversion", "predictedhospitalmortality"])
apache = apache[apache["apacheversion"].astype(str).str.upper().str.strip() == "IVA"].copy()
apache["apache_pred_mortality"] = pd.to_numeric(apache["predictedhospitalmortality"], errors="coerce")
apache.loc[apache["apache_pred_mortality"] < 0, "apache_pred_mortality"] = np.nan
apache = apache.drop_duplicates("patientunitstayid")[["patientunitstayid", "apache_pred_mortality"]]
out = out.merge(apache, on="patientunitstayid", how="left").drop(columns="unitdischargeoffset")
out.to_parquet(INTERIM / "outcomes.parquet", index=False)

summ = out.groupby("role").agg(
    sejours=("patientunitstayid", "size"), mortalite=("y_mortality", "mean"),
    aki_evaluable=("aki_ascertained", "mean"), aki_taux=("y_aki", "mean"),
    los_gt7=("y_los_gt7", "mean"), apache_dispo=("apache_pred_mortality", lambda s: s.notna().mean()))
display(summ.round(4))
asc = out.groupby("hospitalid")["aki_ascertained"].mean()
print(f"Ascertainment de l'AKI par hôpital : min {asc.min():.3f}, max {asc.max():.3f}, écart-type {asc.std():.3f}")

,sejours,mortalite,aki_evaluable,aki_taux,los_gt7,apache_dispo
role,,,,,,
dev,66094,0.0937,0.8079,0.0365,0.1106,0.8805
locked,24640,0.0823,0.7948,0.0323,0.0877,0.8533


Ascertainment de l'AKI par hôpital : min 0.000, max 0.974, écart-type 0.119


## Étape 9 — Tenseurs bruts par hôpital

Pour chaque séjour (T = 24 pas horaires, F variables) :

| Tenseur | Contenu | Remarque |
|---|---|---|
| `X_raw` | valeur moyenne de l'heure, **NaN si absente** | pas de forward-fill ni de normalisation à ce stade |
| `C` | nombre d'événements d'observation dans l'heure | inclut les GCS « non évaluables » |
| `M` | `C > 0` | M = 1 n'implique pas toujours une valeur (sédation) |
| `Delta` | heures depuis l'observation précédente (strictement avant t) ; t + 1 si jamais observée | convention compatible GRU-D |
| `S_num` | âge, taille, poids, sexe (bruts, NaN conservés) | taille/poids impossibles mis à NaN |
| `dx_code` | diagnostic d'admission APACHE (index dans `dx_vocab`) | le top-15 est choisi plus tard, par pli |

La température est incluse si au moins 50 % des séjours (split train, hôpitaux de développement) en ont une mesure.

In [10]:
# ============================================================
# ÉTAPE 9 — TENSEURS BRUTS PAR HÔPITAL (X_raw, M, C, Delta, S)
# ============================================================
cohort = pd.read_parquet(INTERIM / "cohort_final.parquet")
splits = pd.read_parquet(SPLITS / "patient_splits.parquet").set_index("patientunitstayid")
outcomes = pd.read_parquet(INTERIM / "outcomes.parquet").set_index("patientunitstayid")

# ---- décision température (sans label : couverture sur le train des hôpitaux de dév.) ----
dev_train = splits.index[(splits["role"] == "dev") & (splits["split"] == "train")]
temp_stays = pd.read_parquet(INTERIM / "temp_hourly.parquet", columns=["patientunitstayid"])["patientunitstayid"]
temp_cov = float(pd.Index(temp_stays.unique()).isin(dev_train).sum() / max(len(dev_train), 1))
decisions = {"include_temperature": bool(temp_cov >= TEMP_MIN_COVERAGE),
             "temperature_coverage_dev_train": round(temp_cov, 4),
             "temperature_min_coverage": TEMP_MIN_COVERAGE}
save_json(decisions, INTERIM / "decisions.json")
print(f"Couverture température (train, dév.) : {temp_cov:.1%} -> incluse : {decisions['include_temperature']}")

FEATURES = (list(LAB_VARS) + list(VITAL_VARS) + list(GCS_VARS) + ["gcs_unable"]
            + (["temperature"] if decisions["include_temperature"] else []) + ["urine_output"])
def block_of(f):
    return "lab" if f in LAB_VARS else ("monitor" if f in VITAL_VARS else "nursing")
def group_of(f):
    if f in LAB_VARS:
        return "bloodgas" if f in BLOODGAS_LABS else "routine"
    if f in VITAL_VARS:
        return "monitor"
    return {"gcs_unable": "gcs", "temperature": "temperature", "urine_output": "urine"}.get(f, "gcs")
feat_meta = pd.DataFrame({"idx": range(len(FEATURES)), "name": FEATURES,
                          "block": [block_of(f) for f in FEATURES],
                          "group": [group_of(f) for f in FEATURES],
                          "log": [f in LOG_FEATURES for f in FEATURES]})
feat_meta.to_csv(REPORTS / "features.csv", index=False)
F = len(FEATURES)

# ---- remplissage des tenseurs, table par table (économe en mémoire) ----
stays = cohort["patientunitstayid"].to_numpy()
N = len(stays)
stay_index, feat_index = pd.Index(stays), pd.Index(FEATURES)
X_raw = np.full((N, T, F), np.nan, dtype=np.float32)
C = np.zeros((N, T, F), dtype=np.uint16)
names = ["lab_hourly", "vital_hourly", "gcs_hourly", "urine_hourly"] + (
    ["temp_hourly"] if decisions["include_temperature"] else [])
for n in names:
    df = pd.read_parquet(INTERIM / f"{n}.parquet")
    fi = feat_index.get_indexer(df["var"].astype(str))
    si = stay_index.get_indexer(df["patientunitstayid"])
    ok = (fi >= 0) & (si >= 0)          # si < 0 : séjour d'un hôpital exclu à l'étape 6
    fi, si = fi[ok], si[ok]
    ti = df["hr"].to_numpy()[ok].clip(0, T - 1)
    X_raw[si, ti, fi] = df["value"].to_numpy(dtype=np.float32)[ok]
    C[si, ti, fi] = np.minimum(df["n_obs"].to_numpy()[ok], 65535)
    del df
M = (C > 0).astype(np.uint8)
assert np.all(M[~np.isnan(X_raw)] == 1), "valeur présente sans événement d'observation"

def compute_delta(M):
    n, T_, F_ = M.shape
    D = np.zeros((n, T_, F_), dtype=np.uint8)
    last = np.full((n, F_), -1, dtype=np.int16)
    for t in range(T_):
        D[:, t, :] = np.where(last < 0, t + 1, t - last)
        last = np.where(M[:, t, :] == 1, t, last)
    return D
Delta = compute_delta(M)

# ---- variables statiques brutes ----
S_NUM_COLS = ["age", "height", "weight", "gender_female"]
height = cohort["admissionheight"].where(cohort["admissionheight"].between(*HEIGHT_RANGE))
weight = cohort["admissionweight"].where(cohort["admissionweight"].between(*WEIGHT_RANGE))
gender = cohort["gender"].map({"Male": 0.0, "Female": 1.0})
S_num = np.column_stack([cohort["age_num"], height, weight, gender]).astype(np.float32)
dx = cohort["apacheadmissiondx"].fillna("").astype(str).str.strip().replace("", "missing")
dx_vocab = sorted(dx.unique())
dx_code = pd.Index(dx_vocab).get_indexer(dx).astype(np.int32)

save_json({"features": FEATURES, "block": feat_meta["block"].tolist(), "group": feat_meta["group"].tolist(),
           "log": feat_meta["log"].tolist(), "T": T, "bin_minutes": BIN_MIN,
           "static_num": S_NUM_COLS, "dx_vocab": dx_vocab}, DATA_DIR / "features.json")

# ---- sauvegarde par hôpital (on repart d'un dossier vide) ----
for old in CLIENTS.glob("hospital_*.npz"):
    old.unlink()
SPLIT_CODE = {"train": 0, "val": 1, "test": 2}
sp = splits.loc[stays]
oc = outcomes.loc[stays]
hosp_arr = cohort["hospitalid"].to_numpy()
manifest = []
for h in np.sort(np.unique(hosp_arr)):
    idx = np.flatnonzero(hosp_arr == h)
    split_code = sp["split"].map(SPLIT_CODE).to_numpy()[idx].astype(np.int8)
    np.savez_compressed(
        CLIENTS / f"hospital_{h}.npz",
        stay_id=stays[idx], split=split_code,
        X_raw=X_raw[idx], M=M[idx], C=C[idx], Delta=Delta[idx],
        S_num=S_num[idx], dx_code=dx_code[idx],
        y_mortality=oc["y_mortality"].to_numpy()[idx].astype(np.int8),
        y_aki=oc["y_aki"].to_numpy()[idx].astype(np.float32),
        aki_ascertained=oc["aki_ascertained"].to_numpy()[idx].astype(bool),
        y_los_gt7=oc["y_los_gt7"].to_numpy()[idx].astype(np.int8),
        los_icu_days=oc["los_icu_days"].to_numpy()[idx].astype(np.float32),
        apache_pred=oc["apache_pred_mortality"].to_numpy()[idx].astype(np.float32),
    )
    manifest.append({"hospitalid": int(h), "role": sp["role"].iloc[idx[0]], "fold": int(sp["fold"].iloc[idx[0]]),
                     "n_stays": len(idx), "n_train": int((split_code == 0).sum()),
                     "n_val": int((split_code == 1).sum()), "n_test": int((split_code == 2).sum()),
                     "mortality": float(oc["y_mortality"].to_numpy()[idx].mean())})
manifest = pd.DataFrame(manifest)
manifest.to_csv(CLIENTS / "manifest.csv", index=False)

print(f"{len(manifest)} hôpitaux sauvegardés | {N} séjours | {F} variables temporelles | T = {T}")
display(feat_meta)

Couverture température (train, dév.) : 94.7% -> incluse : True
156 hôpitaux sauvegardés | 90734 séjours | 29 variables temporelles | T = 24


,idx,name,block,group,log
0,0,lactate,lab,bloodgas,True
1,1,creatinine,lab,routine,True
2,2,BUN,lab,routine,True
3,3,sodium,lab,routine,False
4,4,potassium,lab,routine,False
5,5,bicarbonate,lab,routine,False
6,6,glucose,lab,routine,True
7,7,WBC x 1000,lab,routine,True
8,8,Hgb,lab,routine,False
9,9,Hct,lab,routine,False


## Étape 10 — Normalisation par pli et disponibilité par hôpital (sans label)

Chaque hôpital calcule, **sur son split train uniquement**, des statistiques suffisantes (effectifs, sommes, sommes des carrés, comptes de diagnostics) : c'est exactement ce qu'un client enverrait au serveur en fédéré.

* **Mode `global`** (par défaut) : le serveur agrège les statistiques des hôpitaux d'entraînement du pli. Un lactate à 4 mmol/l garde la même valeur normalisée dans tous les hôpitaux.
* **Mode `local`** : chaque hôpital se normalise avec ses propres statistiques (y compris un hôpital tenu à l'écart, sur son split train sans label).
* **Disponibilité `avail`** : une variable est disponible dans un hôpital si au moins 5 % des séjours de son split train l'ont observée. `avail = 0` signale une absence structurelle (module non utilisé), à distinguer d'une absence clinique.

Les variables statiques manquantes sont imputées par la **moyenne** du pli (calculable en fédéré, contrairement à la médiane).

In [11]:
# ============================================================
# ÉTAPE 10 — NORMALISATION PAR PLI + DISPONIBILITÉ PAR HÔPITAL (SANS LABEL)
# ============================================================
feats = load_json(DATA_DIR / "features.json")
protocol = load_json(SPLITS / "protocol.json")
manifest = pd.read_csv(CLIENTS / "manifest.csv")
FEATS = feats["features"]
F = len(FEATS)
log_mask = np.array(feats["log"], dtype=bool)
n_dx = len(feats["dx_vocab"])
missing_code = feats["dx_vocab"].index("missing") if "missing" in feats["dx_vocab"] else -1

def transform(x):
    """Transformation appliquée avant normalisation (identique dans src/opfl_data.py)."""
    x = x.astype(np.float64)
    x[..., log_mask] = np.log1p(np.clip(x[..., log_mask], 0, None))
    return x

# 1) statistiques suffisantes locales (split train de chaque hôpital)
local, coverage = {}, {}
for h in manifest["hospitalid"]:
    d = np.load(CLIENTS / f"hospital_{h}.npz")
    tr = d["split"] == 0
    x = transform(d["X_raw"][tr]); obs = ~np.isnan(x)
    s = d["S_num"][tr].astype(np.float64)
    local[int(h)] = {"n": obs.sum((0, 1)), "sum": np.nansum(x, (0, 1)), "sumsq": np.nansum(x ** 2, (0, 1)),
                     "s_n": (~np.isnan(s)).sum(0), "s_sum": np.nansum(s, 0), "s_sumsq": np.nansum(s ** 2, 0),
                     "dx": np.bincount(d["dx_code"][tr], minlength=n_dx)}
    coverage[int(h)] = d["M"][tr].any(axis=1).mean(0)

def mean_std(n, s, ss):
    n = np.asarray(n, float)
    mean = np.divide(s, n, out=np.zeros_like(n), where=n > 0)
    var = np.divide(ss, n, out=np.zeros_like(n), where=n > 0) - mean ** 2
    std = np.sqrt(np.clip(var, 0, None))
    std = np.where((n < 2) | (std < 1e-6), 1.0, std)
    return mean, std

def aggregate(hospitals):
    agg = {k: sum(local[int(h)][k] for h in hospitals) for k in local[int(hospitals[0])]}
    xm, xs = mean_std(agg["n"], agg["sum"], agg["sumsq"])
    sm, ss_ = mean_std(agg["s_n"], agg["s_sum"], agg["s_sumsq"])
    dx_counts = agg["dx"].copy()
    if missing_code >= 0:
        dx_counts[missing_code] = 0
    top = [int(c) for c in np.argsort(-dx_counts, kind="stable")[:15] if dx_counts[c] > 0]
    return {"x_mean": xm.tolist(), "x_std": xs.tolist(), "x_count": agg["n"].astype(int).tolist(),
            "s_mean": sm.tolist(), "s_std": ss_.tolist(),
            "dx_top_codes": top, "dx_top_names": [feats["dx_vocab"][c] for c in top],
            "dx_missing_code": int(missing_code)}

dev = manifest.loc[manifest["role"] == "dev", "hospitalid"].astype(int).tolist()
locked = manifest.loc[manifest["role"] == "locked", "hospitalid"].astype(int).tolist()
plans = {str(k): (sorted(set(dev) - set(protocol["folds"][str(k)])), protocol["folds"][str(k)])
         for k in range(protocol["n_folds"])}
plans["final"] = (sorted(dev), sorted(locked))

for old in NORMS.glob("*.json"):
    old.unlink()
for name, (train_h, heldout_h) in plans.items():
    assert set(train_h).isdisjoint(heldout_h)
    norm = {"fold": name, "train_hospitals": train_h, "heldout_hospitals": heldout_h,
            "computed_from": "split train des hôpitaux d'entraînement uniquement",
            "features": FEATS, **aggregate(train_h)}
    save_json(norm, NORMS / f"fold_{name}.json")

# 2) statistiques locales + disponibilité (mode "local", indicateur avail)
local_stats = {}
for h, st in local.items():
    m, s = mean_std(st["n"], st["sum"], st["sumsq"])
    local_stats[str(h)] = {"n": st["n"].astype(int).tolist(), "mean": m.tolist(), "std": s.tolist(),
                           "coverage": np.round(coverage[h], 4).tolist(),
                           "avail": (coverage[h] >= AVAIL_MIN_COVERAGE).astype(int).tolist()}
save_json(local_stats, NORMS / "local_stats.json")

avail = pd.DataFrame({h: v["avail"] for h, v in local_stats.items()}, index=FEATS).T
avail.index = avail.index.astype(int)
avail.insert(0, "role", manifest.set_index("hospitalid").loc[avail.index, "role"].to_numpy())
avail.to_csv(REPORTS / "availability_by_hospital.csv", index_label="hospitalid")

ex = load_json(NORMS / "fold_0.json")
print("Fichiers écrits :", sorted(p.name for p in NORMS.glob("*.json")))
display(pd.DataFrame({"variable": FEATS, "log": feats["log"],
                      "moyenne_pli0": np.round(ex["x_mean"], 3), "ecart_type_pli0": np.round(ex["x_std"], 3),
                      "n_obs_pli0": ex["x_count"],
                      "hopitaux_non_dispo": [int((avail[f] == 0).sum()) for f in FEATS]}))
print("Top diagnostics (pli 0) :", ex["dx_top_names"][:5], "...")

Fichiers écrits : ['fold_0.json', 'fold_1.json', 'fold_2.json', 'fold_3.json', 'fold_4.json', 'fold_final.json', 'local_stats.json']


,variable,log,moyenne_pli0,ecart_type_pli0,n_obs_pli0,hopitaux_non_dispo
0,lactate,True,1.189,0.532,20672,12
1,creatinine,True,0.840,0.421,58049,0
2,BUN,True,3.099,0.685,57643,0
3,sodium,False,138.367,6.310,64403,0
4,potassium,False,4.113,0.687,71678,0
5,bicarbonate,False,23.162,5.062,55244,5
6,glucose,True,4.931,0.390,60105,0
7,WBC x 1000,True,2.482,0.499,51439,0
8,Hgb,False,10.697,2.316,62240,0
9,Hct,False,32.137,6.785,61192,0


Top diagnostics (pli 0) : ['Sepsis, pulmonary', 'CVA, cerebrovascular accident/stroke', 'Infarction, acute myocardial (MI)', 'CABG alone, coronary artery bypass grafting', 'CHF, congestive heart failure'] ...


## Étape 11 — Module de chargement `src/opfl_data.py`

Cette cellule écrit le module que les notebooks suivants (baselines, OP-FL) importeront.
Elle doit être la seule source de vérité pour la transformation des données en entrée de modèle.

In [12]:
# %%writefile ../src/opfl_data.py
# """
# opfl_data — chargement des données OP-FL V1 (version 1.1).

# Généré par notebooks/01_donnees_v1.ipynb (étape 11) : modifier le notebook, pas ce fichier.

# Transformation au chargement :
# - X : log1p (variables asymétriques) -> z-score (stats du pli, ou stats locales sans label)
#       -> forward-fill -> valeurs jamais observées = 0 (moyenne).
# - M ∈ {0,1} ; C -> log1p(C) ; Delta -> Delta / T. Aucune statistique nécessaire.
# - S : imputation par la moyenne du pli, z-score de l'âge, de la taille et du poids,
#       sexe en 0/1, diagnostic d'admission en one-hot (top-15 du pli + autre + manquant).
# - avail : (F,) disponibilité de chaque variable dans l'hôpital (1 = module utilisé),
#       calculée sans label sur le split train de l'hôpital.
# """
# import json
# from functools import lru_cache
# from pathlib import Path
# import numpy as np

# SPLIT_CODES = {"train": 0, "val": 1, "test": 2}
# MIN_LOCAL_COUNT = 20          # en dessous, le mode local retombe sur la stat du pli


# def _json(path):
#     return json.loads(Path(path).read_text(encoding="utf-8"))


# @lru_cache(maxsize=None)
# def load_features(data_dir):
#     return _json(Path(data_dir) / "features.json")


# def load_protocol(data_dir):
#     return _json(Path(data_dir) / "splits" / "protocol.json")


# def load_normalizer(data_dir, fold):
#     """fold : 0..n_folds-1 (développement) ou 'final' (test verrouillé)."""
#     name = "fold_final.json" if str(fold) == "final" else f"fold_{int(fold)}.json"
#     return _json(Path(data_dir) / "normalizers" / name)


# def fold_hospitals(data_dir, fold):
#     """Retourne (hôpitaux d'entraînement, hôpitaux tenus à l'écart) pour un pli."""
#     norm = load_normalizer(data_dir, fold)
#     return norm["train_hospitals"], norm["heldout_hospitals"]


# @lru_cache(maxsize=None)
# def _local_stats(data_dir):
#     return _json(Path(data_dir) / "normalizers" / "local_stats.json")


# def availability(data_dir, hospital_id):
#     return np.asarray(_local_stats(str(data_dir))[str(int(hospital_id))]["avail"], np.float32)


# def forward_fill(x):
#     x = x.copy()
#     for t in range(1, x.shape[1]):
#         cur = x[:, t, :]
#         x[:, t, :] = np.where(np.isnan(cur), x[:, t - 1, :], cur)
#     return x


# def build_static(S_num, dx_code, norm):
#     s = S_num.astype(np.float32).copy()
#     mean = np.asarray(norm["s_mean"], np.float32)
#     std = np.asarray(norm["s_std"], np.float32)
#     s = np.where(np.isnan(s), mean, s)
#     s[:, :3] = (s[:, :3] - mean[:3]) / std[:3]
#     top = list(norm["dx_top_codes"])
#     pos = {c: i for i, c in enumerate(top)}
#     col = np.array([pos.get(int(c), len(top)) for c in dx_code], dtype=np.int64)
#     col[dx_code == norm["dx_missing_code"]] = len(top) + 1
#     oh = np.zeros((len(dx_code), len(top) + 2), np.float32)
#     oh[np.arange(len(dx_code)), col] = 1.0
#     return np.concatenate([s, oh], axis=1)


# def static_names(data_dir, norm):
#     feats = load_features(str(data_dir))
#     return feats["static_num"] + [f"dx_{n}" for n in norm["dx_top_names"]] + ["dx_other", "dx_missing"]


# def load_client(data_dir, hospital_id, norm, mode="global"):
#     """Charge un hôpital prêt pour un modèle. norm = load_normalizer(data_dir, fold)."""
#     data_dir = Path(data_dir)
#     feats = load_features(str(data_dir))
#     d = np.load(data_dir / "clients" / f"hospital_{int(hospital_id)}.npz")
#     log_mask = np.asarray(feats["log"], bool)

#     x = d["X_raw"].astype(np.float32)
#     x[..., log_mask] = np.log1p(np.clip(x[..., log_mask], 0, None))
#     mean = np.asarray(norm["x_mean"], np.float32)
#     std = np.asarray(norm["x_std"], np.float32)
#     if mode == "local":
#         loc = _local_stats(str(data_dir))[str(int(hospital_id))]
#         ok = np.asarray(loc["n"]) >= MIN_LOCAL_COUNT
#         mean = np.where(ok, np.asarray(loc["mean"], np.float32), mean)
#         std = np.where(ok, np.asarray(loc["std"], np.float32), std)
#     elif mode != "global":
#         raise ValueError("mode doit valoir 'global' ou 'local'")
#     x = (x - mean) / std
#     x = np.nan_to_num(forward_fill(x), nan=0.0).astype(np.float32)

#     T = feats["T"]
#     return {
#         "hospital_id": int(hospital_id),
#         "stay_id": d["stay_id"], "split": d["split"],
#         "X": x,
#         "M": d["M"].astype(np.float32),
#         "C": np.log1p(d["C"].astype(np.float32)),
#         "Delta": d["Delta"].astype(np.float32) / T,
#         "S": build_static(d["S_num"], d["dx_code"], norm),
#         "avail": availability(data_dir, hospital_id),
#         "y_mortality": d["y_mortality"].astype(np.float32),
#         "y_aki": d["y_aki"], "aki_ascertained": d["aki_ascertained"],
#         "y_los_gt7": d["y_los_gt7"].astype(np.float32),
#         "apache_pred": d["apache_pred"],
#     }


# def select(client, split):
#     """Sous-ensemble d'un client : split = 'train', 'val', 'test' ou une liste."""
#     codes = [SPLIT_CODES[s] for s in ([split] if isinstance(split, str) else split)]
#     keep = np.isin(client["split"], codes)
#     return {k: (v[keep] if isinstance(v, np.ndarray) and v.ndim > 0 and len(v) == len(keep) else v)
#             for k, v in client.items()}

Overwriting ../src/opfl_data.py


## Étape 12 — Contrôles de cohérence

Contrôles du protocole (disjonction des plis, des rôles, des patients), de la qualité des hôpitaux retenus, des tenseurs bruts et des données chargées dans les deux modes de normalisation. La cellule échoue si un contrôle ne passe pas.

In [13]:
# ============================================================
# ÉTAPE 12 — CONTRÔLES DE COHÉRENCE
# ============================================================
sys.path.insert(0, str(SRC_DIR.resolve()))
import opfl_data
importlib.reload(opfl_data)
opfl_data.load_features.cache_clear(); opfl_data._local_stats.cache_clear()

protocol = load_json(SPLITS / "protocol.json")
manifest = pd.read_csv(CLIENTS / "manifest.csv")
splits = pd.read_parquet(SPLITS / "patient_splits.parquet")
cohort = pd.read_parquet(INTERIM / "cohort_final.parquet")
quality = pd.read_csv(REPORTS / "hospital_quality.csv").set_index("hospitalid")
feats = load_json(DATA_DIR / "features.json")
F = len(feats["features"])

checks = []
def check(name, ok):
    checks.append({"contrôle": name, "ok": bool(ok)})

dev = set(manifest.loc[manifest["role"] == "dev", "hospitalid"])
locked = set(manifest.loc[manifest["role"] == "locked", "hospitalid"])
folds = [set(v) for v in protocol["folds"].values()]
check("Développement et test verrouillé disjoints", dev.isdisjoint(locked))
check("Plis disjoints et couvrant tout le développement",
      sum(map(len, folds)) == len(dev) and set().union(*folds) == dev)
check("Test verrouillé = hôpitaux 'locked'", set(protocol["locked_test_hospitals"]) == locked)
check("Aucun patient dans deux hôpitaux ou deux splits",
      splits.groupby("uniquepid")[["hospitalid", "split"]].nunique().max().max() == 1)
check("Splits = cohorte finale (mêmes séjours)", set(splits["patientunitstayid"]) == set(cohort["patientunitstayid"]))
check("Hôpitaux retenus : labo et monitorage >= seuil",
      quality.loc[list(dev | locked), "included"].all()
      and (quality.loc[list(dev | locked), ["cov_routine_lab_any", "cov_heartrate"]] >= CORE_MIN_COVERAGE).all().all())
check("Aucun hôpital exclu dans le protocole",
      set(quality.index[~quality["included"]]).isdisjoint(dev | locked))
for k in range(protocol["n_folds"]):
    tr, ho = opfl_data.fold_hospitals(DATA_DIR, k)
    check(f"Pli {k} : statistiques sans les hôpitaux tenus à l'écart",
          set(tr).isdisjoint(ho) and set(ho) == set(protocol["folds"][str(k)]) and set(tr) == dev - set(ho))
tr, ho = opfl_data.fold_hospitals(DATA_DIR, "final")
check("Pli final : entraînement = développement, écart = test verrouillé", set(tr) == dev and set(ho) == locked)

norm0 = opfl_data.load_normalizer(DATA_DIR, 0)
seen, raw_ok, load_ok, local_ok, delta_ok, avail_ok = [], True, True, True, True, True
for h in manifest["hospitalid"]:
    d = np.load(CLIENTS / f"hospital_{h}.npz")
    seen.append(d["stay_id"])
    raw_ok &= set(np.unique(d["M"])) <= {0, 1} and np.all(d["M"][~np.isnan(d["X_raw"])] == 1)
    raw_ok &= set(np.unique(d["split"])) <= {0, 1, 2} and d["X_raw"].shape[1:] == (T, F)
    delta_ok &= int(d["Delta"].min()) >= 0 and int(d["Delta"].max()) <= T
    g = opfl_data.load_client(DATA_DIR, h, norm0, mode="global")
    load_ok &= all(np.isfinite(g[k]).all() for k in ["X", "M", "C", "Delta", "S"])
    avail_ok &= g["avail"].shape == (F,) and set(np.unique(g["avail"])) <= {0.0, 1.0}
    l = opfl_data.load_client(DATA_DIR, h, norm0, mode="local")
    local_ok &= np.isfinite(l["X"]).all()
seen = np.concatenate(seen)
check("Chaque séjour sauvegardé exactement une fois",
      len(seen) == len(np.unique(seen)) == len(cohort) and set(seen) == set(cohort["patientunitstayid"]))
check("Tenseurs bruts : M binaire, valeur => M = 1, splits valides, formes", raw_ok)
check("Delta dans [0, T]", delta_ok)
check("Chargement mode global : aucune valeur NaN/Inf", load_ok)
check("Chargement mode local : aucune valeur NaN/Inf", local_ok)
check("Indicateur de disponibilité avail binaire, de taille F", avail_ok)

res = pd.DataFrame(checks)
display(res)

# ---- comparaison avec l'ancienne extraction (82 664 séjours, 69 hôpitaux) ----
OLD_COUNTS = {"potassium": 77008, "creatinine": 76137, "sodium": 76120, "BUN": 75805, "glucose": 75664,
              "Hct": 74914, "Hgb": 74604, "WBC x 1000": 73698, "platelets x 1000": 73182,
              "bicarbonate": 71490, "FiO2": 35664, "total bilirubin": 34566, "paO2": 33374,
              "paCO2": 32969, "pH": 32610, "lactate": 23452}
pres, n_dev = np.zeros(F), 0
for h in sorted(dev):
    d = np.load(CLIENTS / f"hospital_{h}.npz")
    pres += d["M"].any(axis=1).sum(0); n_dev += len(d["stay_id"])
comp = pd.DataFrame({"variable": feats["features"], "presence_nouvelle (dév.)": np.round(pres / n_dev, 3)})
comp["presence_ancienne"] = comp["variable"].map({k: round(v / 82664, 3) for k, v in OLD_COUNTS.items()})
display(comp)

if not res["ok"].all():
    raise AssertionError("Au moins un contrôle a échoué : voir le tableau ci-dessus.")
print("Tous les contrôles passent.")

,contrôle,ok
0,Développement et test verrouillé disjoints,True
1,Plis disjoints et couvrant tout le développement,True
2,Test verrouillé = hôpitaux 'locked',True
3,Aucun patient dans deux hôpitaux ou deux splits,True
4,Splits = cohorte finale (mêmes séjours),True
5,Hôpitaux retenus : labo et monitorage >= seuil,True
6,Aucun hôpital exclu dans le protocole,True
7,Pli 0 : statistiques sans les hôpitaux tenus à...,True
8,Pli 1 : statistiques sans les hôpitaux tenus à...,True
9,Pli 2 : statistiques sans les hôpitaux tenus à...,True


,variable,presence_nouvelle (dév.),presence_ancienne
0,lactate,0.276,0.284
1,creatinine,0.919,0.921
2,BUN,0.914,0.917
3,sodium,0.918,0.921
4,potassium,0.930,0.932
5,bicarbonate,0.859,0.865
6,glucose,0.912,0.915
7,WBC x 1000,0.891,0.892
8,Hgb,0.902,0.902
9,Hct,0.907,0.906


Tous les contrôles passent.


## Étape 13 — Couverture par hôpital et fiche de données

* `reports/coverage_by_hospital.csv` : pour chaque hôpital et chaque variable, part des séjours ayant au moins une observation en 24 h.
* `reports/availability_by_hospital.csv` : indicateur de disponibilité (étape 10).
* `DATA_CARD.md` : fiche de données pour la section *Methods* de l'article.

In [14]:
# ============================================================
# ÉTAPE 13 — COUVERTURE PAR HÔPITAL ET FICHE DE DONNÉES
# ============================================================
protocol = load_json(SPLITS / "protocol.json")
manifest = pd.read_csv(CLIENTS / "manifest.csv")
feats = load_json(DATA_DIR / "features.json")
decisions = load_json(INTERIM / "decisions.json")
outcomes = pd.read_parquet(INTERIM / "outcomes.parquet")
flow_df = pd.read_csv(REPORTS / "flowchart.csv")
quality = pd.read_csv(REPORTS / "hospital_quality.csv")
avail = pd.read_csv(REPORTS / "availability_by_hospital.csv", index_col="hospitalid")
FEATS = feats["features"]

rows = []
for h in manifest["hospitalid"]:
    d = np.load(CLIENTS / f"hospital_{h}.npz")
    rows.append(d["M"].any(axis=1).mean(0))
cov = pd.DataFrame(rows, columns=FEATS, index=manifest["hospitalid"]).round(4)
cov.insert(0, "role", manifest["role"].to_numpy())
cov.to_csv(REPORTS / "coverage_by_hospital.csv")
summary = pd.DataFrame({"couverture_moy": cov[FEATS].mean(), "min": cov[FEATS].min(), "max": cov[FEATS].max(),
                        "ecart_type": cov[FEATS].std(),
                        "hopitaux_non_dispo": (avail[FEATS] == 0).sum()}).round(3)
summary.to_csv(REPORTS / "feature_coverage_summary.csv")
display(summary)

qc = {p.stem: load_json(p) for p in sorted(INTERIM.glob("qc_*.json"))}
lab_summ = outcomes.groupby("role").agg(
    sejours=("patientunitstayid", "size"), mortalite=("y_mortality", "mean"),
    aki_evaluable=("aki_ascertained", "mean"), aki_taux=("y_aki", "mean"), los_gt7=("y_los_gt7", "mean")).round(4)
folds_tab = manifest.assign(groupe=np.where(manifest["fold"] >= 0, "pli " + manifest["fold"].astype(str), "test verrouillé")
                            ).groupby("groupe").agg(hopitaux=("hospitalid", "size"), sejours=("n_stays", "sum"))
feat_tab = pd.read_csv(REPORTS / "features.csv").merge(
    summary[["couverture_moy", "hopitaux_non_dispo"]].rename_axis("name").reset_index(), on="name")
excl = quality.loc[~quality["included"], ["hospitalid", "role", "n_stays", "cov_routine_lab_any", "cov_heartrate"]].round(3)

def md_table(df):
    df = df.reset_index() if df.index.name else df
    head = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return head + "\n".join("| " + " | ".join(map(str, r)) + " |" for r in df.itertuples(index=False))

card = f"""# Fiche de données — OP-FL V1.1 (eICU-CRD)

Générée le {datetime.now():%Y-%m-%d %H:%M} par `notebooks/01_donnees_v1.ipynb`. Hash du protocole : `{protocol['hash']}`.

## Cohorte
{md_table(flow_df)}

Hôpitaux exclus pour absence structurelle de laboratoire ou de monitorage (< {CORE_MIN_COVERAGE:.0%} des séjours) :

{md_table(excl) if len(excl) else "aucun"}

## Protocole
- Développement : hôpitaux >= {protocol['criteria']['min_stays_dev']} séjours, {protocol['n_folds']} plis leave-hospitals-out équilibrés en taille.
- Test final verrouillé : hôpitaux de {protocol['criteria']['min_stays_locked']} à {protocol['criteria']['min_stays_dev'] - 1} séjours, évalués une seule fois.
- Au sein de chaque hôpital : split par patient {protocol['split_ratios']}, stratifié sur la mortalité, seed {protocol['seed']}.
- Hôpital tenu à l'écart : split train = adaptation sans label ; val + test = évaluation.

{md_table(folds_tab)}

## Variables temporelles (fenêtre 0-24 h, pas horaire, T = {feats['T']})
{md_table(feat_tab[['idx', 'name', 'block', 'group', 'log', 'couverture_moy', 'hopitaux_non_dispo']].round(3))}

Température : couverture {decisions['temperature_coverage_dev_train']:.1%} (train, dév.) -> incluse : {decisions['include_temperature']}.
`hopitaux_non_dispo` : nombre d'hôpitaux où la variable est observée chez moins de {AVAIL_MIN_COVERAGE:.0%} des séjours (absence structurelle).

## Définitions
- `X_raw` : moyenne horaire des valeurs plausibles (somme pour la diurèse), NaN si absente.
- `C` : nombre d'événements d'observation dans l'heure (une saisie GCS « non évaluable » compte ; doublons °C/°F de température retirés).
- `M = C > 0`.
- `Delta` : heures depuis l'observation précédente strictement avant t ; t + 1 si jamais observée.
- `S` : âge, taille, poids, sexe + diagnostic d'admission (top-15 du pli + autre + manquant).
- `avail` : disponibilité de la variable dans l'hôpital (>= {AVAIL_MIN_COVERAGE:.0%} des séjours du split train).
- Normalisation au chargement (`src/opfl_data.py`) : log1p pour {sorted(set(f for f, l in zip(FEATS, feats['log']) if l))},
  z-score avec les statistiques du split train des hôpitaux d'entraînement du pli (mode global) ou de l'hôpital lui-même (mode local),
  puis forward-fill ; valeur jamais observée = 0.

## Labels
{md_table(lab_summ)}

- Mortalité : décès hospitalier.
- AKI : pic de créatinine [24 h, 7 j] / minimum sur 24 h >= 2, défini si dosée aux deux périodes.
- APACHE IVa : référence clinique uniquement, jamais une entrée.

## Contrôle qualité
```
{json.dumps(qc, indent=1, ensure_ascii=False)[:5000]}
```
"""
(DATA_DIR / "DATA_CARD.md").write_text(card, encoding="utf-8")
print("Fiche écrite :", (DATA_DIR / "DATA_CARD.md").resolve())

,couverture_moy,min,max,ecart_type,hopitaux_non_dispo
lactate,0.259,0.000,0.736,0.145,12
creatinine,0.915,0.618,0.989,0.063,0
BUN,0.913,0.569,0.991,0.065,0
sodium,0.916,0.590,0.993,0.063,0
potassium,0.916,0.176,0.993,0.088,0
bicarbonate,0.866,0.000,0.993,0.200,5
glucose,0.907,0.136,0.993,0.093,0
WBC x 1000,0.878,0.458,0.987,0.081,0
Hgb,0.885,0.161,0.991,0.098,0
Hct,0.890,0.582,0.990,0.073,0


Fiche écrite : C:\Users\Stryve\Desktop\StryveSolution\Imagerie-Phd\P.H.D\New Impl\FUALA-w_28-06-2026\OP-FL-V2\data_v1\DATA_CARD.md
